# Biohub Cell Tracking - Data Inspection
Run this on Kaggle (add the competition dataset as input). It dumps the exact structure of one `.zarr` image volume and its paired `.geff` tracking graph, plus sparsity/division stats. Paste the full stdout back.

Enable **Internet** in notebook settings if the zarr>=3 install is needed.

In [1]:
# geff files are zarr v3 (note the zarr.json + c/0/0 sharded chunks). Need zarr>=3.
# REQUIRES INTERNET ON (Settings -> Internet) so pip can install.
import importlib, subprocess, sys

def pip_install(*pkgs):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print('pip install FAILED for', pkgs, '\n', r.stderr[-800:])
    return r.returncode == 0

# install zarr (v3) if missing or too old -- import is guarded so a missing module won't crash
try:
    import zarr
    if int(zarr.__version__.split('.')[0]) < 3:
        pip_install('zarr>=3'); importlib.reload(zarr)
except ModuleNotFoundError:
    print('zarr missing -> installing...')
    pip_install('zarr>=3')
    import zarr
print('zarr', zarr.__version__)

try:
    import geff; print('geff', getattr(geff, '__version__', '?'))
except ModuleNotFoundError:
    geff = None; print('geff not installed (raw-zarr fallback is used anyway, fine)')

zarr missing -> installing...
zarr 3.2.1
geff not installed (raw-zarr fallback is used anyway, fine)


In [2]:
import os, glob, json, numpy as np
from collections import Counter

# 1) Show the REAL directory tree under /kaggle/input (up to a few levels)
def show_tree(root, depth=0, maxdepth=3, maxitems=40):
    try: entries = sorted(os.listdir(root))
    except Exception as e: print('  '*depth, '(cannot list:', e, ')'); return
    for e in entries[:maxitems]:
        p = os.path.join(root, e)
        tag = '/' if os.path.isdir(p) else ''
        print('  '*depth + '- ' + e + tag)
        if os.path.isdir(p) and depth < maxdepth and not (e.endswith('.zarr') or e.endswith('.geff')):
            show_tree(p, depth+1, maxdepth, maxitems)
    if len(entries) > maxitems:
        print('  '*depth + f'... (+{len(entries)-maxitems} more)')

print('=== /kaggle/input tree ===')
show_tree('/kaggle/input')

# 2) Recursively find .geff and .zarr directories anywhere under /kaggle/input
def find_dirs(root, suffix):
    hits = []
    for dp, dns, _ in os.walk(root):
        for d in list(dns):
            if d.endswith(suffix):
                hits.append(os.path.join(dp, d))
                dns.remove(d)  # don't descend into zarr/geff internals
    return sorted(hits)

geffs = find_dirs('/kaggle/input', '.geff')
zarrs = find_dirs('/kaggle/input', '.zarr')
print('\n# .geff found:', len(geffs), '| # .zarr found:', len(zarrs))
print('first geffs:'); [print('  ', p) for p in geffs[:6]]
print('first zarrs:'); [print('  ', p) for p in zarrs[:6]]

# split into train (has paired geff) vs test (zarr only), by parent-dir heuristic
def parent(p): return os.path.dirname(p)
geff_parents = {parent(p) for p in geffs}
train_zarrs = [z for z in zarrs if parent(z) in geff_parents]
test_zarrs  = [z for z in zarrs if parent(z) not in geff_parents]
print('\ntrain .zarr (paired w/ geff):', len(train_zarrs), '| test .zarr:', len(test_zarrs))

assert geffs, 'No .geff found under /kaggle/input -- check that the competition data is added as input.'
assert zarrs, 'No .zarr found under /kaggle/input.'

def du_gb(path):
    tot = 0
    for dp, _, fs in os.walk(path):
        for f in fs:
            try: tot += os.path.getsize(os.path.join(dp, f))
            except OSError: pass
    return tot / 1e9
print(f'\nsize of one .zarr ({os.path.basename(zarrs[0])}): {du_gb(zarrs[0]):.2f} GB')

# pick the sample pair used by later cells
sample_geff = geffs[0]
# find the .zarr sharing the same basename stem as the geff, else first train zarr
stem = os.path.basename(sample_geff)[:-5]  # strip .geff
sample_zarr = next((z for z in zarrs if os.path.basename(z)[:-5] == stem), (train_zarrs or zarrs)[0])
print('\nUSING PAIR:')
print('  geff =', sample_geff)
print('  zarr =', sample_zarr)

=== /kaggle/input tree ===
- competitions/
  - biohub-cell-tracking-during-development/
    - sample_submission.csv
    - test/
      - 44b6_0113de3b.zarr/
      - 44b6_0b24845f.zarr/
      - 6bba_05b6850b.zarr/
      - 6bba_05db0fb1.zarr/
    - train/
      - 44b6_0113de3b.geff/
      - 44b6_0113de3b.zarr/
      - 44b6_0b24845f.geff/
      - 44b6_0b24845f.zarr/
      - 44b6_0c582fdc.geff/
      - 44b6_0c582fdc.zarr/
      - 44b6_0db75fae.geff/
      - 44b6_0db75fae.zarr/
      - 44b6_12dfb391.geff/
      - 44b6_12dfb391.zarr/
      - 44b6_144b256d.geff/
      - 44b6_144b256d.zarr/
      - 44b6_1574802b.geff/
      - 44b6_1574802b.zarr/
      - 44b6_18ced818.geff/
      - 44b6_18ced818.zarr/
      - 44b6_1d530831.geff/
      - 44b6_1d530831.zarr/
      - 44b6_24264f12.geff/
      - 44b6_24264f12.zarr/
      - 44b6_267148e4.geff/
      - 44b6_267148e4.zarr/
      - 44b6_2a2eff9f.geff/
      - 44b6_2a2eff9f.zarr/
      - 44b6_2f31fc2f.geff/
      - 44b6_2f31fc2f.zarr/
      - 44b6_33b596

In [3]:
# 2) Structural dump via raw zarr.json (works regardless of zarr version)
def read_zarr_json(d):
    p = os.path.join(d, 'zarr.json')            # v3
    if os.path.exists(p):
        return json.load(open(p)), 3
    for name, kind in [('.zarray', 'array'), ('.zgroup', 'group')]:  # v2
        p = os.path.join(d, name)
        if os.path.exists(p):
            meta = json.load(open(p))
            attrs_p = os.path.join(d, '.zattrs')
            if os.path.exists(attrs_p): meta['attributes'] = json.load(open(attrs_p))
            meta['node_type'] = kind
            return meta, 2
    return None, None

def walk_zarr(root, prefix='', depth=0, maxdepth=4):
    meta, ver = read_zarr_json(root)
    name = prefix or os.path.basename(root)
    if meta is None:
        return
    nt = meta.get('node_type', 'group')
    if nt == 'array':
        shape = meta.get('shape')
        dt = meta.get('data_type') or meta.get('dtype')
        chunks = None
        try: chunks = meta['chunk_grid']['configuration']['chunk_shape']
        except Exception: chunks = meta.get('chunks')
        print(f"{'  '*depth}[ARR] {name}  shape={shape} dtype={dt} chunks={chunks}")
    else:
        print(f"{'  '*depth}[GRP] {name}")
    attrs = meta.get('attributes', {})
    if attrs:
        s = json.dumps(attrs)
        print(f"{'  '*depth}      attrs: {s[:600]}{'...' if len(s)>600 else ''}")
    if depth >= maxdepth: return
    for child in sorted(os.listdir(root)):
        cp = os.path.join(root, child)
        if os.path.isdir(cp) and read_zarr_json(cp)[0] is not None:
            walk_zarr(cp, child, depth+1, maxdepth)

print('=== IMAGE (.zarr) STRUCTURE ===')
walk_zarr(sample_zarr)
print('\n=== TRACKING GRAPH (.geff) STRUCTURE ===')
walk_zarr(sample_geff)

=== IMAGE (.zarr) STRUCTURE ===
[GRP] 44b6_0113de3b.zarr
      attrs: {"multiscales": [{"version": "0.5", "axes": [{"name": "T", "type": "time", "unit": "second"}, {"name": "Z", "type": "space", "unit": "micrometer"}, {"name": "Y", "type": "space", "unit": "micrometer"}, {"name": "X", "type": "space", "unit": "micrometer"}], "datasets": [{"path": "0", "coordinateTransformations": [{"type": "scale", "scale": [1.0, 1.625, 0.40625, 0.40625]}]}], "name": "0"}], "image_statistics": {"quantiles": {"0.0": 15.0, "0.001": 26.222222222222225, "0.01": 38.0, "0.1": 75.0, "0.9": 497.0, "0.99": 1478.0, "0.999": 2145.000000039654, "1.0": 4319.0}}}
  [ARR] 0  shape=[100, 64, 256, 256] dtype=uint16 chunks=[1, 64, 256, 256]

=== TRACKING GRAPH (.geff) STRUCTURE ===
[GRP] 44b6_0113de3b.geff
      attrs: {"geff": {"geff_version": "1.1", "directed": true, "axes": [{"name": "t", "type": "time", "unit": null, "min": 0.0, "max": 75.0, "scale": 1.0, "scaled_unit": null, "offset": null}, {"name": "z", "type": "

In [4]:
# 3) Load the image array to confirm shape/dtype/value range
img_root = zarr.open(sample_zarr, mode='r')
def find_arrays(g, path=''):
    out = []
    try:
        for k in g.keys():
            sub = g[k]; sp = f'{path}/{k}' if path else k
            if hasattr(sub, 'shape'): out.append((sp, sub))
            else: out += find_arrays(sub, sp)
    except Exception: pass
    return out
arrs = find_arrays(img_root)
if not arrs and hasattr(img_root, 'shape'):
    arrs = [('<root>', img_root)]
print('arrays found:', [(p, a.shape, str(a.dtype)) for p, a in arrs])
path0, arr = max(arrs, key=lambda pa: int(np.prod(pa[1].shape)))
print(f'\nmain array: {path0}  shape={arr.shape} dtype={arr.dtype} chunks={arr.chunks}')
# sample the first timepoint's volume for value range
try:
    idx = tuple(0 if i < arr.ndim-3 else slice(None) for i in range(arr.ndim))
    vol = np.asarray(arr[idx]); print('one-timepoint volume shape:', vol.shape)
    print('value range: min=%.3f max=%.3f mean=%.3f' % (vol.min(), vol.max(), vol.mean()))
except Exception as e:
    print('sampling failed:', e)

arrays found: [('0', (100, 64, 256, 256), 'uint16')]

main array: 0  shape=(100, 64, 256, 256) dtype=uint16 chunks=(1, 64, 256, 256)
one-timepoint volume shape: (64, 256, 256)
value range: min=12.000 max=2950.000 mean=219.034


In [5]:
# 4) Read the tracking graph: node props, edges, sparsity, divisions
def load_graph_raw(geff_path):
    g = zarr.open(geff_path, mode='r')
    node_ids = np.asarray(g['nodes/ids'][:])
    props = {}
    try: pkeys = list(g['nodes/props'].keys())
    except Exception: pkeys = []
    for k in pkeys:
        for sub in ['values', '']:
            try:
                arr = g[f'nodes/props/{k}/{sub}'] if sub else g[f'nodes/props/{k}']
                props[k] = np.asarray(arr[:]); break
            except Exception: pass
    edges = None
    try: edges = np.asarray(g['edges/ids'][:])
    except Exception as e: print('edge load fail:', e)
    return node_ids, props, edges

node_ids, props, edges = load_graph_raw(sample_geff)
print('sample:', os.path.basename(sample_geff))
print('num nodes:', len(node_ids), '| num edges:', None if edges is None else len(edges))
print('node prop keys:', list(props.keys()))
for k, v in props.items():
    print(f'  prop {k}: dtype={v.dtype} shape={v.shape} sample={v[:5].tolist() if v.ndim==1 else v[:2].tolist()}')

# identify the time prop
tkey = next((k for k in props if k.lower() in ('t','time','frame')), None)
print('\ntime prop =', tkey)
if tkey is not None:
    tvals = props[tkey].astype(int)
    per_frame = Counter(tvals.tolist())
    frames = sorted(per_frame)
    print('num timepoints:', len(frames), '| t range:', frames[0], '->', frames[-1])
    counts = [per_frame[f] for f in frames]
    print('nodes/frame: min=%d max=%d mean=%.1f (this = how SPARSE the GT is)' % (min(counts), max(counts), np.mean(counts)))
    print('first 15 frames counts:', counts[:15])

sample: 44b6_0113de3b.geff
num nodes: 52 | num edges: 50
node prop keys: ['y', 't', 'x', 'z']
  prop y: dtype=int64 shape=(52,) sample=[222, 227, 230, 74, 78]
  prop t: dtype=int64 shape=(52,) sample=[0, 1, 2, 27, 28]
  prop x: dtype=int64 shape=(52,) sample=[249, 251, 253, 73, 75]
  prop z: dtype=int64 shape=(52,) sample=[63, 63, 63, 1, 3]

time prop = t
num timepoints: 52 | t range: 0 -> 75
nodes/frame: min=1 max=1 mean=1.0 (this = how SPARSE the GT is)
first 15 frames counts: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [6]:
# 5) Division stats + lineage structure (out-degree >= 2 = division)
if edges is not None and len(edges):
    src = edges[:, 0]
    out_deg = Counter(src.tolist())
    in_deg = Counter(edges[:, 1].tolist())
    div_nodes = [n for n, d in out_deg.items() if d >= 2]
    print('division nodes (out-deg>=2):', len(div_nodes))
    print('out-degree distribution:', dict(Counter(out_deg.values())))
    print('in-degree  distribution:', dict(Counter(in_deg.values())))
    # typical displacement between linked nodes (helps set linking distance threshold)
    id2idx = {int(i): k for k, i in enumerate(node_ids)}
    coord_keys = [k for k in ['z','y','x'] if k in props]
    if coord_keys and tkey:
        scale = {'z':1.625,'y':0.40625,'x':0.40625}
        disp = []
        for s, t in edges[:2000]:
            if int(s) in id2idx and int(t) in id2idx:
                a, b = id2idx[int(s)], id2idx[int(t)]
                d2 = sum((scale[c]*(props[c][a]-props[c][b]))**2 for c in coord_keys)
                disp.append(d2**0.5)
        if disp:
            disp = np.array(disp)
            print('\nphysical displacement per link (um): median=%.2f p90=%.2f max=%.2f' % (np.median(disp), np.percentile(disp,90), disp.max()))

# 6) Confirm test dataset names (for submission 'dataset' column)
print('\ntest dataset names (dataset column values):')
for p in (test_zarrs or []):
    print('  ', os.path.basename(p).replace('.zarr',''))

division nodes (out-deg>=2): 0
out-degree distribution: {1: 50}
in-degree  distribution: {1: 50}

physical displacement per link (um): median=2.88 p90=4.67 max=7.81

test dataset names (dataset column values):
   44b6_0113de3b
   44b6_0b24845f
   6bba_05b6850b
   6bba_05db0fb1


In [7]:
# 7) DIVISION FREQUENCY across ALL train samples (the ceiling for the 0.1 * division_jaccard term)
# A division = a GT node with out-degree >= 2 (mother -> two daughters at t+1). Rare, but this cell
# tells us: how many exist, in which specimen, and the mother->daughter geometry (to set post-hoc gates).
SCALE_UM = {'z': 1.625, 'y': 0.40625, 'x': 0.40625}

def sample_division_info(geff_path):
    """Return (n_nodes, n_edges, n_div, [ (motherdisp1, motherdisp2, inter_daughter_um, angle_deg) ... ])."""
    node_ids, props, edges = load_graph_raw(geff_path)
    if edges is None or len(edges) == 0:
        return len(node_ids), 0, 0, []
    src = edges[:, 0]
    out_children = {}
    for s, t in edges:
        out_children.setdefault(int(s), []).append(int(t))
    id2idx = {int(i): k for k, i in enumerate(node_ids)}
    def xyz(nid):
        k = id2idx[nid]
        return np.array([SCALE_UM['z']*props['z'][k], SCALE_UM['y']*props['y'][k], SCALE_UM['x']*props['x'][k]], float)
    geoms = []
    n_div = 0
    for m, kids in out_children.items():
        if len(kids) < 2:
            continue
        n_div += 1
        if m not in id2idx:
            continue
        pm = xyz(m)
        # take the two daughters (if >2, first two) for geometry
        d = [xyz(k) for k in kids[:2] if k in id2idx]
        if len(d) == 2:
            v1, v2 = d[0]-pm, d[1]-pm
            n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
            inter = float(np.linalg.norm(d[0]-d[1]))
            ang = float(np.degrees(np.arccos(np.clip(np.dot(v1, v2)/(n1*n2+1e-9), -1, 1)))) if n1>1e-6 and n2>1e-6 else np.nan
            geoms.append((float(n1), float(n2), inter, ang))
    return len(node_ids), len(edges), n_div, geoms

tot_nodes = tot_edges = tot_div = 0
per_sample = []          # (name, n_div)
all_geoms = []
by_specimen = Counter()  # divisions per specimen prefix
samples_with_div = 0
for gp in geffs:
    name = os.path.basename(gp)[:-5]
    nn, ne, nd, geoms = sample_division_info(gp)
    tot_nodes += nn; tot_edges += ne; tot_div += nd
    per_sample.append((name, nd))
    all_geoms += geoms
    if nd > 0:
        samples_with_div += 1
        by_specimen[name[:4]] += nd

print(f'=== DIVISION FREQUENCY over {len(geffs)} train samples ===')
print(f'total GT nodes: {tot_nodes:,} | total GT edges: {tot_edges:,} | total division events: {tot_div}')
print(f'samples with >=1 division: {samples_with_div}/{len(geffs)}')
print(f'divisions as fraction of edges: {tot_div/max(tot_edges,1)*100:.3f}%  '
      f'(=> max possible from the 0.1 division term if you nail them all)')
print(f'divisions by specimen: {dict(by_specimen)}')

nd_arr = np.array([nd for _, nd in per_sample])
print(f'\nper-sample division count: min={nd_arr.min()} max={nd_arr.max()} '
      f'mean={nd_arr.mean():.2f} median={np.median(nd_arr):.0f}')
top = sorted(per_sample, key=lambda x: -x[1])[:12]
print('most division-rich samples (use these for local division eval):')
for name, nd in top:
    if nd == 0: break
    print(f'   {name}: {nd} divisions')

if all_geoms:
    g = np.array(all_geoms)  # cols: motherdisp1, motherdisp2, inter_daughter_um, angle_deg
    md = np.concatenate([g[:, 0], g[:, 1]])
    print(f'\nmother->daughter displacement (um): median={np.median(md):.2f} '
          f'p90={np.percentile(md,90):.2f} max={md.max():.2f}')
    print(f'inter-daughter distance   (um): median={np.median(g[:,2]):.2f} '
          f'p90={np.percentile(g[:,2],90):.2f} max={g[:,2].max():.2f}')
    ang = g[:, 3][~np.isnan(g[:, 3])]
    if len(ang):
        print(f'daughter-split angle     (deg): median={np.median(ang):.1f} '
              f'p10={np.percentile(ang,10):.1f} p90={np.percentile(ang,90):.1f}  '
              f'(near 180 = daughters on opposite sides => symmetry gate is discriminative)')
else:
    print('\nno divisions found in any sample (or geometry unavailable).')

=== DIVISION FREQUENCY over 199 train samples ===
total GT nodes: 133,318 | total GT edges: 128,883 | total division events: 151
samples with >=1 division: 87/199
divisions as fraction of edges: 0.117%  (=> max possible from the 0.1 division term if you nail them all)
divisions by specimen: {'44b6': 26, '6bba': 125}

per-sample division count: min=0 max=5 mean=0.76 median=0
most division-rich samples (use these for local division eval):
   6bba_48816121: 5 divisions
   6bba_09961292: 4 divisions
   6bba_afb141ff: 4 divisions
   6bba_cdcfe533: 4 divisions
   6bba_debd7bfa: 4 divisions
   6bba_df673a83: 4 divisions
   6bba_05db0fb1: 3 divisions
   6bba_12665c0e: 3 divisions
   6bba_20852818: 3 divisions
   6bba_4ffd3da3: 3 divisions
   6bba_57b7cc1e: 3 divisions
   6bba_5c039895: 3 divisions

mother->daughter displacement (um): median=5.75 p90=9.07 max=13.53
inter-daughter distance   (um): median=10.57 p90=14.36 max=20.30
daughter-split angle     (deg): median=138.2 p10=88.1 p90=165.1  (